# RL Test Flow Optimization — Notebook 2 of 3: Stage-2 + HPO

**Prerequisite**: NB1 complete + `rl-stage1-results` dataset added as input.

| Step | Description | Est. Time |
|------|-------------|----------|
| 6 | Stage-2: top-2 algos × 3 seeds × 500K steps | ~5-6 h |
| 7 | Optuna HPO: 30 trials × 100K steps | ~2-3 h |

> **After NB2 completes**: Output tab → Create Dataset → name it **`rl-stage2-results`**
> Then open NB3 and add both `rl-stage1-results` AND `rl-stage2-results` as inputs.

## Step 0: Reinstall + Load NB1 Results

In [ ]:
import subprocess, sys, os, json
subprocess.run(['pip', 'install', '-q',
    'stable-baselines3[extra]', 'sb3-contrib', 'gymnasium',
    'optuna', 'mlflow', 'pyarrow', 'matplotlib', 'numpy', 'torch'], check=True)

!rm -rf rl-test-flow-optimization
!git clone https://github.com/rajendarmuddasani/rl-test-flow-optimization.git
os.chdir('rl-test-flow-optimization')
sys.path.insert(0, '.')

import torch, numpy as np, pandas as pd
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable GPU!"}')

# Load NB1 results
NB1_PATH = '/kaggle/input/rl-stage1-results/stage1_results.json'
with open(NB1_PATH) as f:
    nb1 = json.load(f)

baseline_results  = nb1['baselines']
stage1_results    = nb1['stage1']
top2_algos        = nb1['top2_algos']
best_bl           = nb1['best_base_reward']
baseline_df       = pd.DataFrame(baseline_results).T
stage1_df         = pd.DataFrame(stage1_results).T

print(f'NB1 results loaded:')
print(f'  Top-2 algos: {top2_algos}')
print(f'  Best baseline reward: {best_bl:+.2f}')
print(f'  Stage-1 best: {stage1_df["mean_reward"].idxmax()} = {stage1_df["mean_reward"].max():+.2f}')


## Step 6: Stage-2 — Deep Training Top-2 (500K steps × 3 seeds)

3 random seeds to measure variance. RL is stochastic — one run could be lucky.
Mean ± std across seeds is the production-grade measurement standard.

In [ ]:
import time as _time
from src.environment import TestFlowEnv
from src.agent import ALGO_REGISTRY, evaluate_trained_model
from pathlib import Path

STAGE2_STEPS = 500_000
SEEDS = [42, 123, 777]
SAVE_DIR2 = Path('/kaggle/working/rl_stage2')
SAVE_DIR2.mkdir(parents=True, exist_ok=True)

# Copy NB1 models into working dir for reference
import shutil
nb1_models = Path('/kaggle/input/rl-stage1-results')
if nb1_models.exists():
    shutil.copytree(nb1_models, SAVE_DIR2 / 'stage1_ref', dirs_exist_ok=True)

stage2_results = {}

print(f'Stage-2: {STAGE2_STEPS:,} steps × {len(SEEDS)} seeds')
print(f'Algorithms: {top2_algos}')
print('=' * 70)

for algo_name in top2_algos:
    train_fn = ALGO_REGISTRY[algo_name]
    seed_metrics = []
    for seed in SEEDS:
        print(f'\n>>> {algo_name.upper()} seed={seed}')
        env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
        t0 = _time.time()
        model = train_fn(
            env,
            total_timesteps=STAGE2_STEPS,
            output_dir=str(SAVE_DIR2 / f'{algo_name}_seed{seed}'),
            seed=seed,
        )
        elapsed = _time.time() - t0
        metrics = evaluate_trained_model(env, model, n_episodes=500)
        metrics['seed'] = seed
        metrics['train_time_sec'] = round(elapsed, 1)
        seed_metrics.append(metrics)
        print(f'  reward={metrics["mean_reward"]:+.2f} | acc={metrics["accuracy"]:.3f} | {elapsed/60:.1f}m')
    stage2_results[algo_name] = seed_metrics

print(f'\n{"="*60}')
print('Stage-2 Seed Stability')
print(f'{"Algorithm":20s} {"Mean Reward":>12s} {"Std Reward":>12s} {"Mean Acc":>10s}')
print('-'*60)
for algo in top2_algos:
    rewards = [m['mean_reward'] for m in stage2_results[algo]]
    accs    = [m['accuracy']    for m in stage2_results[algo]]
    print(f'{algo:20s} {np.mean(rewards):>+12.2f} {np.std(rewards):>12.3f} {np.mean(accs):>10.3f}')


## Step 7: Optuna HPO — 30 Trials × 100K Steps

Tree-structured Parzen Estimator (TPE) search over:
- `learning_rate`: 1e-5 to 1e-2 (log scale)
- `gamma`: 0.90 to 0.999
- `batch_size`: 32, 64, 128, 256
- `net_arch`: [64,64], [128,128], [256,256], [256,256,128]

AMD/production standard: 30+ trials is the minimum for reliable HPO.

In [ ]:
from src.agent import run_optuna_hpo

best_algo = top2_algos[0]
print(f'Running Optuna HPO for: {best_algo}')
print(f'  Trials: 30  |  Steps/trial: 100,000')
print(f'  Search: lr[1e-5..1e-2], gamma[0.9..0.999], batch_size[32/64/128/256]')

hpo_result = run_optuna_hpo(
    env_cls=TestFlowEnv,
    env_kwargs={'n_tests': 200, 'cost_budget': 50.0, 'time_budget': 120.0},
    algo=best_algo,
    n_trials=30,
    timesteps=100_000,
)

print(f'\n{"="*50}')
print('OPTUNA HPO COMPLETE')
print(f'  Best reward:  {hpo_result["best_value"]:+.2f}')
print(f'  Best params:')
for k, v in hpo_result['best_params'].items():
    print(f'    {k}: {v}')
print(f'  Total trials: {hpo_result["n_trials"]}')


## Save NB2 Outputs

> After this cell: Output tab → Create Dataset → **`rl-stage2-results`**
> Add both `rl-stage1-results` and `rl-stage2-results` as inputs to NB3.

In [ ]:
import json

save_data2 = {
    'baselines':      baseline_results,
    'stage1':         stage1_results,
    'stage2':         stage2_results,
    'hpo':            hpo_result,
    'top2_algos':     top2_algos,
    'best_algo':      best_algo,
    'best_base_reward': float(best_bl),
    'best_params':    hpo_result['best_params'],
}

with open(SAVE_DIR2 / 'stage2_results.json', 'w') as f:
    json.dump(save_data2, f, indent=2, default=str)

print('=== NB2 ARTIFACTS ===')
total = 0
for ff in sorted(SAVE_DIR2.rglob('*')):
    if ff.is_file() and ff.suffix in ['.json', '.zip']:
        print(f'  {str(ff.relative_to(SAVE_DIR2)):60s} {ff.stat().st_size/1e3:6.0f} KB')
        total += ff.stat().st_size
print(f'\nTotal: {total/1e6:.1f} MB')
print('\nNB2 complete!')
print('Next: Output tab → Create Dataset → rl-stage2-results')
print('Then open NB3 and add both rl-stage1-results + rl-stage2-results')
